# 02 特徴量エンジニアリング

このノートブックでは、モデルに渡す特徴量を設計する

今回のゴールは「東京の時系列気象データから、24時間後の天気クラスを予測する」こと

単純に現在の気温・湿度・気圧だけを使うのではなく、以下のような情報を作る

- 直近の値：1時間前、3時間前、24時間前
- 変化量：気圧が下がっているか、湿度が上がっているか
- 移動統計：過去24時間の平均、最大、最小、ばらつき
- 時間情報：季節、時間帯
- 直近の天気：前の天気クラス

天気は「その瞬間の値」だけではなく、直近の変化の流れが重要と仮定する


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NUMERIC_COLUMNS = [
    "temperature",
    "humidity",
    "pressure",
    "precipitation",
    "wind_speed",
    "cloud_amount",
]

CATEGORICAL_COLUMNS = [
    "wind_direction",
]


In [2]:
df = pd.read_csv("./data/all_data_after_eda.csv")
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df

,datetime,temperature,humidity,pressure,precipitation,no_rain,wind_speed,wind_direction,wind_speed_gn,cloud_amount,weather_code,weather_class
0,2021-01-01 02:00:00,0.4,46.0,1008.4,0.0,1,1.5,北西,1,0.0,1.0,1
1,2021-01-01 03:00:00,-0.6,52.0,1008.0,0.0,1,1.1,西北西,1,0.0,1.0,1
2,2021-01-01 04:00:00,0.1,51.0,1007.8,0.0,1,1.8,北西,1,0.0,1.0,1
3,2021-01-01 05:00:00,0.0,53.0,1007.8,0.0,1,0.6,北西,1,0.0,1.0,1
4,2021-01-01 06:00:00,-0.4,58.0,1008.4,0.0,1,1.1,西北西,1,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
35046,2024-12-31 20:00:00,8.4,49.0,1002.9,0.0,1,3.3,北北西,1,0.0,1.0,1
35047,2024-12-31 21:00:00,7.7,51.0,1004.5,0.0,1,3.1,北北西,1,0.0,1.0,1
35048,2024-12-31 22:00:00,6.5,56.0,1005.5,0.0,1,3.1,北西,1,0.0,1.0,1
35049,2024-12-31 23:00:00,6.3,55.0,1006.3,0.0,1,1.9,北西,1,0.0,1.0,1


## 1. 目的変数を作る

`weather_class`を１日後ろにずらして、翌日の天気クラスを目的変数にする

注意点：

- 最終行は翌日の正解がないので欠損になる
- 欠損した最終行は学習対象から除外する

In [3]:
horizon_hours = 24

feature_df = df.copy()
feature_df["target_weather_class"] = feature_df["weather_class"].shift(-horizon_hours)
display(feature_df[["datetime", "weather_class", "target_weather_class"]].head(30))
display(feature_df[["datetime", "weather_class", "target_weather_class"]].tail(30))

,datetime,weather_class,target_weather_class
0,2021-01-01 02:00:00,1,1.0
1,2021-01-01 03:00:00,1,1.0
2,2021-01-01 04:00:00,1,1.0
3,2021-01-01 05:00:00,1,1.0
4,2021-01-01 06:00:00,1,1.0
5,2021-01-01 07:00:00,1,1.0
6,2021-01-01 08:00:00,1,1.0
7,2021-01-01 09:00:00,1,1.0
8,2021-01-01 10:00:00,1,1.0
9,2021-01-01 11:00:00,1,1.0


,datetime,weather_class,target_weather_class
35021,2024-12-30 19:00:00,1,1.0
35022,2024-12-30 20:00:00,1,1.0
35023,2024-12-30 21:00:00,1,1.0
35024,2024-12-30 22:00:00,1,1.0
35025,2024-12-30 23:00:00,1,1.0
35026,2024-12-31 00:00:00,1,1.0
35027,2024-12-31 01:00:00,1,NaN
35028,2024-12-31 02:00:00,1,NaN
35029,2024-12-31 03:00:00,2,NaN
35030,2024-12-31 04:00:00,2,NaN


## 2. ラグ特徴量を作る

ラグ特徴量とは、過去の値を現在の行に持たせる特徴量

例：

- `temperature_lag_1h`: 1時間前の気温
- `pressure_lag_24h`: 24時間前の現地気圧
- `weather_class_lag_24h`: 24時間前の天気

天気は連続的に変化するため、直前や前日の状態が影響する

In [4]:
# 1時間前, 3時間前, 6時間前, 12時間前, 24時間前
lags=(1,3,6,12,24)

for col in NUMERIC_COLUMNS:
    for lag in lags:
        feature_df[f"{col}_lag_{lag}h"] = feature_df[col].shift(lag)
    # データはそのままで、メモリ上でのデータの並び順を整理する（最適化）
    # 警告「PerformanceWarning: DataFrame is highly fragmented. 」対策
    feature_df = feature_df.copy()

feature_df.filter(regex="lag").head(10)

,temperature_lag_1h,temperature_lag_3h,temperature_lag_6h,temperature_lag_12h,temperature_lag_24h,humidity_lag_1h,humidity_lag_3h,humidity_lag_6h,humidity_lag_12h,humidity_lag_24h,...,wind_speed_lag_1h,wind_speed_lag_3h,wind_speed_lag_6h,wind_speed_lag_12h,wind_speed_lag_24h,cloud_amount_lag_1h,cloud_amount_lag_3h,cloud_amount_lag_6h,cloud_amount_lag_12h,cloud_amount_lag_24h
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.4,NaN,NaN,NaN,NaN,46.0,NaN,NaN,NaN,NaN,...,1.5,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
2,-0.6,NaN,NaN,NaN,NaN,52.0,NaN,NaN,NaN,NaN,...,1.1,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
3,0.1,0.4,NaN,NaN,NaN,51.0,46.0,NaN,NaN,NaN,...,1.8,1.5,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
4,0.0,-0.6,NaN,NaN,NaN,53.0,52.0,NaN,NaN,NaN,...,0.6,1.1,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
5,-0.4,0.1,NaN,NaN,NaN,58.0,51.0,NaN,NaN,NaN,...,1.1,1.8,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
6,-0.3,0.0,0.4,NaN,NaN,61.0,53.0,46.0,NaN,NaN,...,0.8,0.6,1.5,NaN,NaN,0.0,0.0,0.0,NaN,NaN
7,2.1,-0.4,-0.6,NaN,NaN,48.0,58.0,52.0,NaN,NaN,...,1.1,1.1,1.1,NaN,NaN,0.0,0.0,0.0,NaN,NaN
8,4.2,-0.3,0.1,NaN,NaN,42.0,61.0,51.0,NaN,NaN,...,1.1,0.8,1.8,NaN,NaN,0.0,0.0,0.0,NaN,NaN
9,6.0,2.1,0.0,NaN,NaN,39.0,48.0,53.0,NaN,NaN,...,1.9,1.1,0.6,NaN,NaN,0.0,0.0,0.0,NaN,NaN


## 3. 差分特徴量

差分特徴量は「どれくらい変化したか」を表す

例：

- 気圧が6時間で下がった
- 湿度が3時間で上がった
- 風速が急に強くなった

特に天気予測では、気圧・湿度・雲量・降水量の変化が重要な手がかりになる

In [5]:
# 1時間, 3時間前, 6時間前, 24時間前
periods=(1,3,6,24)

for col in NUMERIC_COLUMNS:
    for p in periods:
        feature_df[f"{col}_diff_{p}h"] = feature_df[col].diff(p) # df[col] - df[col].shift(p)
    # データはそのままで、メモリ上でのデータの並び順を整理する（最適化）
    # 警告「PerformanceWarning: DataFrame is highly fragmented. 」対策
    feature_df = feature_df.copy()

feature_df.filter(regex="diff").head(10)

,temperature_diff_1h,temperature_diff_3h,temperature_diff_6h,temperature_diff_24h,humidity_diff_1h,humidity_diff_3h,humidity_diff_6h,humidity_diff_24h,pressure_diff_1h,pressure_diff_3h,...,precipitation_diff_6h,precipitation_diff_24h,wind_speed_diff_1h,wind_speed_diff_3h,wind_speed_diff_6h,wind_speed_diff_24h,cloud_amount_diff_1h,cloud_amount_diff_3h,cloud_amount_diff_6h,cloud_amount_diff_24h
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-1.0,NaN,NaN,NaN,6.0,NaN,NaN,NaN,-0.4,NaN,...,NaN,NaN,-0.4,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,0.7,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,-0.2,NaN,...,NaN,NaN,0.7,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,-0.1,-0.4,NaN,NaN,2.0,7.0,NaN,NaN,0.0,-0.6,...,NaN,NaN,-1.2,-0.9,NaN,NaN,0.0,0.0,NaN,NaN
4,-0.4,0.2,NaN,NaN,5.0,6.0,NaN,NaN,0.6,0.4,...,NaN,NaN,0.5,0.0,NaN,NaN,0.0,0.0,NaN,NaN
5,0.1,-0.4,NaN,NaN,3.0,10.0,NaN,NaN,0.6,1.2,...,NaN,NaN,-0.3,-1.0,NaN,NaN,0.0,0.0,NaN,NaN
6,2.4,2.1,1.7,NaN,-13.0,-5.0,2.0,NaN,0.7,1.9,...,0.0,NaN,0.3,0.5,-0.4,NaN,0.0,0.0,0.0,NaN
7,2.1,4.6,4.8,NaN,-6.0,-16.0,-10.0,NaN,0.4,1.7,...,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN
8,1.8,6.3,5.9,NaN,-3.0,-22.0,-12.0,NaN,0.3,1.4,...,0.0,NaN,0.8,1.1,0.1,NaN,0.0,0.0,0.0,NaN
9,2.0,5.9,8.0,NaN,-6.0,-15.0,-20.0,NaN,-0.5,0.2,...,0.0,NaN,-1.3,-0.5,0.0,NaN,0.0,0.0,0.0,NaN


## 4. 移動統計特徴量

移動平均・最大・最小・標準偏差を使うと、短期的なノイズではなく「傾向」を見られる

例:

- 過去24時間の平均湿度
- 過去6時間の最低気圧
- 過去12時間の雲量のばらつき

これにより、モデルは「傾向」を学習しやすくなる

In [6]:
# 3時間単位, 6時間単位, 12時間単位, 24時間単位
windows=(3, 6, 12, 24)

for col in NUMERIC_COLUMNS:
    for window in windows:
        feature_df[f"{col}_mean_{window}h"] = feature_df[col].rolling(window).mean()
        feature_df[f"{col}_std_{window}h"] = feature_df[col].rolling(window).std()
        feature_df[f"{col}_min_{window}h"] = feature_df[col].rolling(window).min()
        feature_df[f"{col}_max_{window}h"] = feature_df[col].rolling(window).max()
    # データはそのままで、メモリ上でのデータの並び順を整理する（最適化）
    # 警告「PerformanceWarning: DataFrame is highly fragmented. 」対策
    feature_df = feature_df.copy()

feature_df.filter(regex="mean|std|min|max").head(10)

,temperature_mean_3h,temperature_std_3h,temperature_min_3h,temperature_max_3h,temperature_mean_6h,temperature_std_6h,temperature_min_6h,temperature_max_6h,temperature_mean_12h,temperature_std_12h,...,cloud_amount_min_6h,cloud_amount_max_6h,cloud_amount_mean_12h,cloud_amount_std_12h,cloud_amount_min_12h,cloud_amount_max_12h,cloud_amount_mean_24h,cloud_amount_std_24h,cloud_amount_min_24h,cloud_amount_max_24h
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.033333,0.513160,-0.6,0.4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.166667,0.378594,-0.6,0.1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-0.100000,0.264575,-0.4,0.1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,-0.233333,0.208167,-0.4,0.0,-0.133333,0.366970,-0.6,0.4,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.466667,1.415392,-0.4,2.1,0.150000,0.989444,-0.6,2.1,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2.000000,2.251666,-0.3,4.2,0.950000,1.838206,-0.4,4.2,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,4.100000,1.951922,2.1,6.0,1.933333,2.678557,-0.4,6.0,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,6.066667,1.900877,4.2,8.0,3.266667,3.413893,-0.4,8.0,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. 時間特徴量を作る

天気には強い季節性がある

例えば、同じ気温15℃でも、春の15℃と秋の15℃では翌日の天気傾向が違う可能性がある

ここでは、月・日・年内通算日を特徴量として追加する

注意するポイント：

- 周期性を表すためにsin/cos変換する
    - 23時と0時、12月と1月が近い季節として扱われやすくなる
    - ふつうに「月」を`1,2,...12`という数字のまま学習させると、以下のように判断してしまう
        - 「1月」と「2月」の差は1（近い）
        - 「12月」と「1月」の差は11（遠い）


In [7]:
feature_df["hour"] = feature_df["datetime"].dt.hour
feature_df["month"] = feature_df["datetime"].dt.month
feature_df["dayofyear"] = feature_df["datetime"].dt.dayofyear

# 周期性を表すためにsin/cos変換する
feature_df["hour_sin"] = np.sin(2 * np.pi * feature_df["hour"] / 24)
feature_df["hour_cos"] = np.cos(2 * np.pi * feature_df["hour"] / 24)
feature_df["month_sin"] = np.sin(2 * np.pi * feature_df["month"] / 12)
feature_df["month_cos"] = np.cos(2 * np.pi * feature_df["month"] / 12)

# データはそのままで、メモリ上でのデータの並び順を整理する（最適化）
# 警告「PerformanceWarning: DataFrame is highly fragmented. 」対策
feature_df = feature_df.copy()

feature_df[["datetime", "hour", "month", "dayofyear", "hour_sin", "month_sin", "month_cos", "hour_cos"]].head()

,datetime,hour,month,dayofyear,hour_sin,month_sin,month_cos,hour_cos
0,2021-01-01 02:00:00,2,1,1,0.500000,0.5,0.866025,8.660254e-01
1,2021-01-01 03:00:00,3,1,1,0.707107,0.5,0.866025,7.071068e-01
2,2021-01-01 04:00:00,4,1,1,0.866025,0.5,0.866025,5.000000e-01
3,2021-01-01 05:00:00,5,1,1,0.965926,0.5,0.866025,2.588190e-01
4,2021-01-01 06:00:00,6,1,1,1.000000,0.5,0.866025,6.123234e-17


## 6. 直近・24時間後の天気クラス

天気はそのまま同じ天気が続く可能性が高い

また別の天気に変化する時に、前の天気が影響する可能性もある

つまり、天気クラスも学習データとして有効になる

In [8]:
# 直前の天気も強い特徴量になる
feature_df["weather_class_lag_1h"] = feature_df["weather_class"].shift(1)
feature_df["weather_class_lag_3h"] = feature_df["weather_class"].shift(3)
feature_df["weather_class_lag_6h"] = feature_df["weather_class"].shift(6)
feature_df["weather_class_lag_24h"] = feature_df["weather_class"].shift(24)

# データはそのままで、メモリ上でのデータの並び順を整理する（最適化）
# 警告「PerformanceWarning: DataFrame is highly fragmented. 」対策
feature_df = feature_df.copy()

feature_df.filter(regex="weather_class_lag").head(10)

,weather_class_lag_1h,weather_class_lag_3h,weather_class_lag_6h,weather_class_lag_24h
0,NaN,NaN,NaN,NaN
1,1.0,NaN,NaN,NaN
2,1.0,NaN,NaN,NaN
3,1.0,1.0,NaN,NaN
4,1.0,1.0,NaN,NaN
5,1.0,1.0,NaN,NaN
6,1.0,1.0,1.0,NaN
7,1.0,1.0,1.0,NaN
8,1.0,1.0,1.0,NaN
9,1.0,1.0,1.0,NaN


## 7. 文字列データを数値データに変換

風向は角度や方位として扱うデータであり、

単純に0,1,2,3... のようなカテゴリ数値として扱うと、北と北北西の距離感がうまく表現できてないことがある

ここでは簡易的に `pd.get_dummies` でカテゴリ変数として扱う


In [9]:
if "wind_direction" in feature_df.columns:
    feature_df = pd.get_dummies(feature_df, columns=["wind_direction"], prefix="wind_dir", dummy_na=True)

feature_df.head()

,datetime,temperature,humidity,pressure,precipitation,no_rain,wind_speed,wind_speed_gn,cloud_amount,weather_code,...,wind_dir_南東,wind_dir_南西,wind_dir_東,wind_dir_東北東,wind_dir_東南東,wind_dir_西,wind_dir_西北西,wind_dir_西南西,wind_dir_静穏,wind_dir_nan
0,2021-01-01 02:00:00,0.4,46.0,1008.4,0.0,1,1.5,1,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
1,2021-01-01 03:00:00,-0.6,52.0,1008.0,0.0,1,1.1,1,0.0,1.0,...,False,False,False,False,False,False,True,False,False,False
2,2021-01-01 04:00:00,0.1,51.0,1007.8,0.0,1,1.8,1,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
3,2021-01-01 05:00:00,0.0,53.0,1007.8,0.0,1,0.6,1,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
4,2021-01-01 06:00:00,-0.4,58.0,1008.4,0.0,1,1.1,1,0.0,1.0,...,False,False,False,False,False,False,True,False,False,False


## 8. 欠損値（NaN）を含む行を削除

様々なデータを作る仮定で出来てしまった欠損値を含む行は削除する

In [10]:
df = df.dropna().reset_index(drop=True)

## 9. 最終的な特徴量テーブルの評価



In [11]:
print("元データ：", df.shape)
print("特徴量テーブル：", feature_df.shape)

display(feature_df.head())

元データ： (35051, 12)
特徴量テーブル： (35051, 191)


,datetime,temperature,humidity,pressure,precipitation,no_rain,wind_speed,wind_speed_gn,cloud_amount,weather_code,...,wind_dir_南東,wind_dir_南西,wind_dir_東,wind_dir_東北東,wind_dir_東南東,wind_dir_西,wind_dir_西北西,wind_dir_西南西,wind_dir_静穏,wind_dir_nan
0,2021-01-01 02:00:00,0.4,46.0,1008.4,0.0,1,1.5,1,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
1,2021-01-01 03:00:00,-0.6,52.0,1008.0,0.0,1,1.1,1,0.0,1.0,...,False,False,False,False,False,False,True,False,False,False
2,2021-01-01 04:00:00,0.1,51.0,1007.8,0.0,1,1.8,1,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
3,2021-01-01 05:00:00,0.0,53.0,1007.8,0.0,1,0.6,1,0.0,1.0,...,False,False,False,False,False,False,False,False,False,False
4,2021-01-01 06:00:00,-0.4,58.0,1008.4,0.0,1,1.1,1,0.0,1.0,...,False,False,False,False,False,False,True,False,False,False


## 10. データの保存

後続のnotebookで同じ特徴量を扱えるように、CSVとして保存する

In [12]:
output_path = Path("./data/features_weather_24h.csv")

feature_df.to_csv(output_path, index=False)
print(output_path)

data\features_weather_24h.csv
